# IFC Flare / Autoland Diagnostics

Use this notebook to diagnose flare and touchdown behavior from IFC CSV logs.

What this notebook does:
- Loads a chosen log (or auto-selects newest)
- Cleans mixed-type rows (including trailing `# end ...` marker)
- Builds flare-focused derived metrics
- Plots overview + flare zoom + control-authority channels
- Produces an automatic issue summary with likely causes
- Adds vertical-acceleration authority margin diagnostics (required vs measured)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (16, 10)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
pd.set_option("display.max_columns", 200)


In [ ]:
# ===== User config =====
LOGS_DIR = Path(r"f:\Kerbal Space Program\Ships\Script\Integrated Flight Computer\logs")
USE_LATEST_LOG = True

# Used only if USE_LATEST_LOG = False
LOG_FILE_PATH = LOGS_DIR / "ifc_log_00000134.csv"

# Detection thresholds (tune as needed)
VS_NEAR_ZERO_THRESH = 0.1     # [m/s]
EARLY_ARREST_GEAR_H = 5.0     # [m] arrest above this is considered early
BALLOON_VS_THRESH = 0.3       # [m/s]
BALLOON_GEAR_H_MIN = 3.0      # [m]
PERSISTENT_TRACK_ERR_DEG = 2.0

ACCEL_SMOOTH_WINDOW = 3      # samples for dVS/dt smoothing in accel-margin plot


In [ ]:
def find_latest_log(logs_dir: Path) -> Path:
    files = sorted(logs_dir.glob("ifc_log_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not files:
        raise FileNotFoundError(f"No IFC log files found in: {logs_dir}")
    return files[0]


def load_ifc_log(csv_path: Path) -> pd.DataFrame:
    # Ignore trailing comment/footer rows (e.g. '# end T+...').
    raw = pd.read_csv(csv_path, comment="#")
    raw.columns = [c.strip() for c in raw.columns]

    # t_s is required for all plots; coerce and drop invalid rows.
    raw["t_s"] = pd.to_numeric(raw["t_s"], errors="coerce")
    df = raw.dropna(subset=["t_s"]).copy()

    # Keep common categorical fields as strings; coerce everything else numeric where possible.
    categorical_cols = {
        "phase", "subphase", "flare_mode", "asc_validity", "status", "craft_name", "cfg_file"
    }
    for col in df.columns:
        if col in categorical_cols:
            df[col] = df[col].fillna("").astype(str)
        else:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.sort_values("t_s").reset_index(drop=True)
    return df


def phase_segments(df: pd.DataFrame, col: str = "phase"):
    segs = []
    if col not in df.columns or df.empty:
        return segs

    start_idx = 0
    prev = df.iloc[0][col]
    for i in range(1, len(df)):
        cur = df.iloc[i][col]
        if cur != prev:
            segs.append((str(prev), df.iloc[start_idx]["t_s"], df.iloc[i - 1]["t_s"]))
            start_idx = i
            prev = cur
    segs.append((str(prev), df.iloc[start_idx]["t_s"], df.iloc[len(df) - 1]["t_s"]))
    return segs


def add_phase_spans(ax, segs, alpha=0.06):
    colors = {
        "APPROACH": "#4c78a8",
        "FLARE": "#f58518",
        "TOUCHDOWN": "#54a24b",
        "ROLLOUT": "#b279a2",
    }
    for phase, t0, t1 in segs:
        c = colors.get(phase, "#999999")
        ax.axvspan(t0, t1, color=c, alpha=alpha)


def safe_col(df: pd.DataFrame, col: str):
    return col in df.columns


In [ ]:
log_path = find_latest_log(LOGS_DIR) if USE_LATEST_LOG else Path(LOG_FILE_PATH)
df = load_ifc_log(log_path)
segs = phase_segments(df, "phase")

print(f"Loaded: {log_path}")
print(f"Rows: {len(df):,}")
print(f"Time span: {df['t_s'].min():.2f}s -> {df['t_s'].max():.2f}s")
if safe_col(df, "craft_name"):
    print(f"Craft: {df['craft_name'].iloc[-1]}")
if safe_col(df, "cfg_file"):
    print(f"Cfg: {df['cfg_file'].iloc[-1]}")

display(df.head(3))


In [ ]:
# Build flare-focused frame and derived metrics.
flare_mask = df["phase"].isin(["FLARE", "TOUCHDOWN"])
flare_df = df.loc[flare_mask].copy()

if flare_df.empty:
    print("No FLARE/TOUCHDOWN rows in this log.")
else:
    flare_df["abs_vs_err"] = flare_df.get("flare_vs_err", np.nan).abs()
    flare_df["abs_fpa_err"] = flare_df.get("flare_fpa_err", np.nan).abs()
    flare_df["abs_pitch_err"] = flare_df.get("flare_pitch_err", np.nan).abs()

    # Event: first near-zero VS during flare/touchdown.
    near_zero = flare_df[np.abs(flare_df["vs_ms"]) <= VS_NEAR_ZERO_THRESH]
    first_near_zero = near_zero.iloc[0] if not near_zero.empty else None

    # Event: balloon peak while still above touchdown region.
    balloon_rows = flare_df[(flare_df["vs_ms"] >= BALLOON_VS_THRESH) & (flare_df["gear_h_m"] >= BALLOON_GEAR_H_MIN)]
    peak_balloon = balloon_rows.loc[balloon_rows["vs_ms"].idxmax()] if not balloon_rows.empty else None

    print("Flare/touchdown summary:")
    print(f"  Rows: {len(flare_df)}")
    print(f"  gear_h range: {flare_df['gear_h_m'].min():.2f} .. {flare_df['gear_h_m'].max():.2f} m")
    print(f"  vs range: {flare_df['vs_ms'].min():.2f} .. {flare_df['vs_ms'].max():.2f} m/s")

    if first_near_zero is not None:
        print(f"  First |VS|<= {VS_NEAR_ZERO_THRESH:.2f}: t={first_near_zero['t_s']:.2f}, gear_h={first_near_zero['gear_h_m']:.2f}, agl={first_near_zero['agl_m']:.2f}")
    else:
        print(f"  No |VS|<= {VS_NEAR_ZERO_THRESH:.2f} event in FLARE/TOUCHDOWN.")

    if peak_balloon is not None:
        print(f"  Balloon peak: t={peak_balloon['t_s']:.2f}, VS={peak_balloon['vs_ms']:.2f}, gear_h={peak_balloon['gear_h_m']:.2f}")
    else:
        print("  No balloon event above configured threshold.")


In [ ]:
# ===== Overview plot =====
fig, axs = plt.subplots(4, 1, sharex=True, figsize=(18, 14))

axs[0].plot(df["t_s"], df["agl_m"], label="AGL [m]", lw=1.8)
axs[0].plot(df["t_s"], df["gear_h_m"], label="Gear Height [m]", lw=1.4)
if safe_col(df, "gs_nom_alt_m"):
    axs[0].plot(df["t_s"], df["gs_nom_alt_m"], "--", label="GS Nom Alt [m]", alpha=0.8)
axs[0].set_ylabel("Height [m]")
axs[0].legend(loc="best")
add_phase_spans(axs[0], segs)

axs[1].plot(df["t_s"], df["vs_ms"], label="VS [m/s]", lw=1.8)
if safe_col(df, "flare_tgt_vs"):
    axs[1].plot(df["t_s"], df["flare_tgt_vs"], "--", label="Flare Target VS", alpha=0.85)
axs[1].axhline(0, color="k", lw=1, alpha=0.5)
axs[1].set_ylabel("VS [m/s]")
axs[1].legend(loc="best")
add_phase_spans(axs[1], segs)

axs[2].plot(df["t_s"], df["ias_ms"], label="IAS [m/s]", lw=1.8)
if safe_col(df, "vapp_ms"):
    axs[2].plot(df["t_s"], df["vapp_ms"], "--", label="Vapp/Vtgt [m/s]", alpha=0.8)
axs[2].set_ylabel("Speed [m/s]")
axs[2].legend(loc="best")
add_phase_spans(axs[2], segs)

axs[3].plot(df["t_s"], df["thr_cmd"], label="Throttle Cmd", lw=1.8)
if safe_col(df, "thr_cur"):
    axs[3].plot(df["t_s"], df["thr_cur"], "--", label="Throttle Cur", alpha=0.8)
if safe_col(df, "flare_thr_floor"):
    axs[3].plot(df["t_s"], df["flare_thr_floor"], ":", label="Flare Thr Floor", alpha=0.9)
axs[3].set_ylabel("Throttle")
axs[3].set_xlabel("Time [s]")
axs[3].legend(loc="best")
add_phase_spans(axs[3], segs)

for ax in axs:
    ax.grid(True, alpha=0.35)

fig.suptitle(f"IFC Overview - {log_path.name}", y=0.995)
plt.tight_layout()
plt.show()


In [ ]:
# ===== Flare zoom =====
if flare_df.empty:
    print("No FLARE/TOUCHDOWN rows to plot.")
else:
    fig, axs = plt.subplots(4, 1, sharex=True, figsize=(18, 15))

    axs[0].plot(flare_df["t_s"], flare_df["gear_h_m"], label="Gear Height [m]", lw=1.8)
    axs[0].plot(flare_df["t_s"], flare_df["agl_m"], label="AGL [m]", lw=1.2, alpha=0.85)
    axs[0].axhline(0, color="k", lw=1, alpha=0.5)
    axs[0].set_ylabel("Height [m]")
    axs[0].legend(loc="best")

    axs[1].plot(flare_df["t_s"], flare_df["vs_ms"], label="VS [m/s]", lw=1.8)
    if safe_col(flare_df, "flare_tgt_vs"):
        axs[1].plot(flare_df["t_s"], flare_df["flare_tgt_vs"], "--", label="Target VS", alpha=0.9)
    axs[1].axhline(0, color="k", lw=1, alpha=0.5)
    axs[1].set_ylabel("VS [m/s]")
    axs[1].legend(loc="best")

    axs[2].plot(flare_df["t_s"], flare_df["actual_fpa_deg"], label="Actual FPA [deg]", lw=1.8)
    if safe_col(flare_df, "flare_fpa_cmd"):
        axs[2].plot(flare_df["t_s"], flare_df["flare_fpa_cmd"], "--", label="Flare Cmd [deg]", alpha=0.9)
    if safe_col(flare_df, "flare_gamma_ref"):
        axs[2].plot(flare_df["t_s"], flare_df["flare_gamma_ref"], ":", label="Gamma Ref [deg]", alpha=0.9)
    axs[2].set_ylabel("FPA [deg]")
    axs[2].legend(loc="best")

    axs[3].plot(flare_df["t_s"], flare_df["pitch_deg"], label="Pitch [deg]", lw=1.8)
    axs[3].plot(flare_df["t_s"], flare_df["aoa_deg"], label="AoA [deg]", lw=1.4)
    if safe_col(flare_df, "aa_dir_pitch_deg"):
        axs[3].plot(flare_df["t_s"], flare_df["aa_dir_pitch_deg"], "--", label="AA Dir Pitch [deg]", alpha=0.9)
    axs[3].set_ylabel("Angle [deg]")
    axs[3].set_xlabel("Time [s]")
    axs[3].legend(loc="best")

    for ax in axs:
        ax.grid(True, alpha=0.35)

    # Mark flare mode transitions on all subplots.
    if safe_col(flare_df, "flare_mode"):
        mode_change = flare_df[flare_df["flare_mode"].fillna("").ne(flare_df["flare_mode"].shift().fillna(""))]
        for _, r in mode_change.iterrows():
            for ax in axs:
                ax.axvline(r["t_s"], color="#1f77b4", ls=":", lw=1.0, alpha=0.75)
            axs[0].text(r["t_s"], axs[0].get_ylim()[1] * 0.9, str(r["flare_mode"]), rotation=90, fontsize=8)

    fig.suptitle(f"Flare/Touchdown Zoom - {log_path.name}", y=0.995)
    plt.tight_layout()
    plt.show()


In [ ]:
# ===== Control authority diagnostics =====
if flare_df.empty:
    print("No FLARE/TOUCHDOWN rows to analyze.")
else:
    has_new_cols = all(c in flare_df.columns for c in ["flare_pitch_in_cmd", "elev_defl_avg_deg", "elev_defl_max_deg", "elev_defl_n"])

    nrows = 3 if has_new_cols else 2
    fig, axs = plt.subplots(nrows, 1, sharex=True, figsize=(18, 4.2 * nrows))

    axs[0].plot(flare_df["t_s"], flare_df["flare_pitch_err"], label="Pitch Err [deg]", lw=1.8)
    axs[0].plot(flare_df["t_s"], flare_df["flare_fpa_err"], label="FPA Err [deg]", lw=1.4)
    axs[0].plot(flare_df["t_s"], flare_df["flare_vs_err"], label="VS Err [m/s]", lw=1.4)
    axs[0].axhline(0, color="k", lw=1, alpha=0.5)
    axs[0].set_ylabel("Tracking Error")
    axs[0].legend(loc="best")
    axs[0].grid(True, alpha=0.35)

    axs[1].plot(flare_df["t_s"], flare_df["flare_auth_limited"], label="Auth Limited", lw=1.8)
    if safe_col(flare_df, "flare_thr_floor"):
        axs[1].plot(flare_df["t_s"], flare_df["flare_thr_floor"], label="Thr Floor", lw=1.2)
    axs[1].set_ylabel("Latch / Floor")
    axs[1].legend(loc="best")
    axs[1].grid(True, alpha=0.35)

    if has_new_cols:
        axs[2].plot(flare_df["t_s"], flare_df["flare_pitch_in_cmd"], label="Pitch Input Cmd [-1..1]", lw=1.6)
        axs[2].plot(flare_df["t_s"], flare_df["elev_defl_avg_deg"], label="Elev Defl Avg |deg|", lw=1.4)
        axs[2].plot(flare_df["t_s"], flare_df["elev_defl_max_deg"], label="Elev Defl Max |deg|", lw=1.4)
        axs[2].set_ylabel("Cmd / Deflection")
        axs[2].legend(loc="best")
        axs[2].grid(True, alpha=0.35)

    axs[-1].set_xlabel("Time [s]")
    fig.suptitle(f"Control Authority Diagnostics - {log_path.name}", y=0.995)
    plt.tight_layout()
    plt.show()

    if has_new_cols:
        display(flare_df[["t_s", "phase", "flare_mode", "gear_h_m", "vs_ms", "flare_pitch_in_cmd", "elev_defl_avg_deg", "elev_defl_max_deg", "elev_defl_n"]].tail(20))


## TECS Energy Diagnostics

This section visualizes TECS energy behavior during `FLARE` + `TOUCHDOWN`.

Interpretation guide:
- `flare_tecs_et_err` (total-energy error) should generally stay bounded and trend toward zero.
- `flare_tecs_eb_err` (energy-balance error) should also stay bounded; prolonged divergence indicates pitch/throttle split is not closing.
- `flare_gamma_unsat` vs `flare_fpa_cmd` shows command limiting/slew effects.
- `flare_gamma_eb_term` shows how strongly the balance loop is pushing gamma.

"Good" behavior is smooth, bounded errors with no long one-sided drift.
"Bad" behavior is monotonic growth in magnitude, large oscillations, or errors staying large while commands are at limits.


In [ ]:
# ===== TECS / energy plots =====
if flare_df.empty:
    print("No FLARE/TOUCHDOWN rows to analyze.")
else:
    required_cols = [
        "flare_tecs_et_err",
        "flare_tecs_eb_err",
        "flare_tecs_h_ref",
        "flare_tecs_v_ref",
        "flare_gamma_ref",
        "flare_gamma_eb_term",
        "flare_gamma_unsat",
        "flare_fpa_cmd",
        "ias_ms",
        "gear_h_m",
        "thr_cmd",
        "flare_thr_floor",
    ]

    missing = [c for c in required_cols if c not in flare_df.columns]
    if missing:
        print("Missing TECS columns in this log:", missing)
    else:
        g = 9.81
        tecs = flare_df.copy()

        # Approximate energies from logged states using controller height when available.
        h_col = "flare_ctrl_h_m" if ("flare_ctrl_h_m" in tecs.columns and tecs["flare_ctrl_h_m"].notna().any()) else "gear_h_m"
        print(f"TECS energy vertical state: {h_col}")
        tecs["et_ref_est"] = 0.5 * tecs["flare_tecs_v_ref"]**2 + g * tecs["flare_tecs_h_ref"]
        tecs["et_now_est"] = 0.5 * tecs["ias_ms"]**2 + g * tecs[h_col]
        tecs["eb_ref_est"] = g * tecs["flare_tecs_h_ref"] - 0.5 * tecs["flare_tecs_v_ref"]**2
        tecs["eb_now_est"] = g * tecs[h_col] - 0.5 * tecs["ias_ms"]**2

        fig, axs = plt.subplots(4, 1, sharex=True, figsize=(18, 15))

        # 1) Error channels
        axs[0].plot(tecs["t_s"], tecs["flare_tecs_et_err"], label="Et Error", lw=1.8)
        axs[0].plot(tecs["t_s"], tecs["flare_tecs_eb_err"], label="Eb Error", lw=1.8)
        axs[0].axhline(0, color="k", lw=1, alpha=0.5)
        axs[0].set_ylabel("Error [m^2/s^2]")
        axs[0].set_title("TECS Error Channels")
        axs[0].legend(loc="best")
        axs[0].grid(True, alpha=0.35)

        # 2) Estimated total energy levels
        axs[1].plot(tecs["t_s"], tecs["et_ref_est"], label="Et Ref (est)", lw=1.6)
        axs[1].plot(tecs["t_s"], tecs["et_now_est"], label="Et Now (est)", lw=1.6)
        axs[1].set_ylabel("Et [m^2/s^2]")
        axs[1].set_title("Estimated Total Energy")
        axs[1].legend(loc="best")
        axs[1].grid(True, alpha=0.35)

        # 3) Gamma decomposition
        axs[2].plot(tecs["t_s"], tecs["flare_gamma_ref"], label="Gamma Ref", lw=1.5)
        axs[2].plot(tecs["t_s"], tecs["flare_gamma_eb_term"], label="Gamma Eb Term", lw=1.5)
        axs[2].plot(tecs["t_s"], tecs["flare_gamma_unsat"], label="Gamma Unsat", lw=1.5)
        axs[2].plot(tecs["t_s"], tecs["flare_fpa_cmd"], label="Gamma/FPA Cmd", lw=1.8)
        axs[2].axhline(0, color="k", lw=1, alpha=0.5)
        axs[2].set_ylabel("Gamma [deg]")
        axs[2].set_title("Gamma Command Decomposition")
        axs[2].legend(loc="best")
        axs[2].grid(True, alpha=0.35)

        # 4) Throttle-side TECS outputs
        axs[3].plot(tecs["t_s"], tecs["thr_cmd"], label="Throttle Cmd", lw=1.8)
        axs[3].plot(tecs["t_s"], tecs["flare_thr_floor"], label="Flare Thr Floor", lw=1.5)
        if "flare_auth_limited" in tecs.columns:
            axs[3].plot(tecs["t_s"], tecs["flare_auth_limited"], label="Auth Limited", lw=1.2)
        axs[3].set_ylabel("Throttle / Latch")
        axs[3].set_xlabel("Time [s]")
        axs[3].set_title("Throttle Channel")
        axs[3].legend(loc="best")
        axs[3].grid(True, alpha=0.35)

        fig.suptitle(f"TECS Energy Diagnostics - {log_path.name}", y=0.995)
        plt.tight_layout()
        plt.show()

        # Compact numeric summary for quick checks.
        tecs_summary = {
            "Et err mean": tecs["flare_tecs_et_err"].mean(),
            "Et err abs max": tecs["flare_tecs_et_err"].abs().max(),
            "Eb err mean": tecs["flare_tecs_eb_err"].mean(),
            "Eb err abs max": tecs["flare_tecs_eb_err"].abs().max(),
            "Gamma cmd min": tecs["flare_fpa_cmd"].min(),
            "Gamma cmd max": tecs["flare_fpa_cmd"].max(),
        }
        display(pd.Series(tecs_summary).round(3))


## Vertical Acceleration Authority Margin

This plot compares:
- `req_up_a`: upward acceleration required to reach `VS_tgt` before touchdown
- `meas_up_a`: measured upward acceleration (`d(VS)/dt`)
- `a_margin = meas_up_a - req_up_a`

Interpretation:
- `a_margin < 0` while descending means the aircraft is not generating enough upward acceleration to track target VS.
- Sustained negative margin during flare usually means authority/energy/timing mismatch.


In [ ]:
# ===== Vertical acceleration authority margin =====
if flare_df.empty:
    print("No FLARE/TOUCHDOWN rows to analyze.")
else:
    needed = ["t_s", "vs_ms", "flare_tgt_vs"]
    missing = [c for c in needed if c not in flare_df.columns]
    if missing:
        print("Missing columns for accel-margin diagnostics:", missing)
    else:
        av = flare_df.sort_values("t_s").copy()
        t = av["t_s"].to_numpy(dtype=float)
        vs = av["vs_ms"].to_numpy(dtype=float)
        vs_tgt = av["flare_tgt_vs"].to_numpy(dtype=float)
        h_col = "flare_ctrl_h_m" if ("flare_ctrl_h_m" in av.columns and av["flare_ctrl_h_m"].notna().any()) else "gear_h_m"
        print(f"Accel-margin vertical state: {h_col}")
        h = np.maximum(av[h_col].to_numpy(dtype=float), 0.5)

        # Prefer logger-provided required accel when available.
        if "flare_req_up_a" in av.columns and av["flare_req_up_a"].notna().any():
            req_up_a = av["flare_req_up_a"].to_numpy(dtype=float)
        else:
            req_up_a = (vs * vs - vs_tgt * vs_tgt) / (2.0 * h)

        # Measured vertical acceleration: d(VS)/dt.
        meas_up_a_raw = np.gradient(vs, t)
        w = max(int(ACCEL_SMOOTH_WINDOW), 1)
        meas_up_a = pd.Series(meas_up_a_raw).rolling(window=w, center=True, min_periods=1).mean().to_numpy()

        av["req_up_a"] = req_up_a
        av["meas_up_a_raw"] = meas_up_a_raw
        av["meas_up_a"] = meas_up_a
        av["a_margin"] = av["meas_up_a"] - av["req_up_a"]

        focus = (av["vs_ms"] < -0.2) & (av[h_col] > BALLOON_GEAR_H_MIN)
        if focus.any():
            neg_frac = (av.loc[focus, "a_margin"] < 0).mean() * 100.0
            print(f"Negative accel margin while descending above {BALLOON_GEAR_H_MIN:.1f} m gear_h: {neg_frac:.1f}%")
        else:
            print("No focus rows (descending above configured gear-height floor).")

        fig, axs = plt.subplots(3, 1, sharex=True, figsize=(18, 12))

        axs[0].plot(av["t_s"], av["req_up_a"], label="Required Up Accel [m/s^2]", lw=1.8)
        axs[0].plot(av["t_s"], av["meas_up_a"], label=f"Measured Up Accel [m/s^2] (smooth={w})", lw=1.8)
        axs[0].axhline(0, color="k", lw=1, alpha=0.5)
        axs[0].set_ylabel("Accel [m/s^2]")
        axs[0].set_title("Required vs Measured Vertical Acceleration")
        axs[0].legend(loc="best")
        axs[0].grid(True, alpha=0.35)

        axs[1].plot(av["t_s"], av["a_margin"], label="Accel Margin (meas - req)", lw=1.8)
        axs[1].fill_between(av["t_s"], av["a_margin"], 0, where=av["a_margin"] < 0, alpha=0.2, color="red")
        axs[1].axhline(0, color="k", lw=1, alpha=0.5)
        axs[1].set_ylabel("Margin [m/s^2]")
        axs[1].set_title("Vertical Acceleration Margin")
        axs[1].legend(loc="best")
        axs[1].grid(True, alpha=0.35)

        axs[2].plot(av["t_s"], av["vs_ms"], label="VS [m/s]", lw=1.8)
        axs[2].plot(av["t_s"], av["flare_tgt_vs"], "--", label="Target VS [m/s]", alpha=0.9)
        axs[2].axhline(0, color="k", lw=1, alpha=0.5)
        axs[2].set_ylabel("VS [m/s]")
        axs[2].set_xlabel("Time [s]")
        axs[2].set_title("Context: VS Tracking")
        axs[2].legend(loc="best")
        axs[2].grid(True, alpha=0.35)

        fig.suptitle(f"Vertical Acceleration Margin - {log_path.name}", y=0.995)
        plt.tight_layout()
        plt.show()

        view_cols = ["t_s", "phase", "flare_mode", h_col, "vs_ms", "flare_tgt_vs", "req_up_a", "meas_up_a", "a_margin"]
        display(av[view_cols].tail(20))


In [ ]:
# ===== Automatic issue summary =====
issues = []
notes = []

if flare_df.empty:
    print("No FLARE/TOUCHDOWN segment found; cannot run flare diagnostics.")
else:
    # 1) Early sink arrest (near-zero VS while still high above runway)
    near_zero = flare_df[np.abs(flare_df["vs_ms"]) <= VS_NEAR_ZERO_THRESH]
    if not near_zero.empty:
        nz = near_zero.iloc[0]
        if nz["gear_h_m"] >= EARLY_ARREST_GEAR_H:
            issues.append(f"Early sink arrest: first |VS|<={VS_NEAR_ZERO_THRESH:.2f} at gear_h={nz['gear_h_m']:.2f} m (t={nz['t_s']:.2f}).")
        else:
            notes.append(f"Near-zero VS event occurs low: gear_h={nz['gear_h_m']:.2f} m (t={nz['t_s']:.2f}).")
    else:
        notes.append("No near-zero VS event observed during FLARE/TOUCHDOWN.")

    # 2) Balloon detection
    balloon_rows = flare_df[(flare_df["vs_ms"] >= BALLOON_VS_THRESH) & (flare_df["gear_h_m"] >= BALLOON_GEAR_H_MIN)]
    if not balloon_rows.empty:
        peak = balloon_rows.loc[balloon_rows["vs_ms"].idxmax()]
        issues.append(f"Balloon detected: VS peak {peak['vs_ms']:.2f} m/s at gear_h={peak['gear_h_m']:.2f} m (t={peak['t_s']:.2f}).")
    else:
        notes.append("No significant balloon above thresholds.")

    # 3) Persistent pitch tracking miss
    if safe_col(flare_df, "flare_pitch_err"):
        bad_pitch = flare_df[flare_df["flare_pitch_err"] <= -PERSISTENT_TRACK_ERR_DEG]
        frac_bad = len(bad_pitch) / max(len(flare_df), 1)
        if frac_bad >= 0.20:
            issues.append(f"Persistent pitch tracking miss: {frac_bad*100:.1f}% of flare rows have pitch_err <= -{PERSISTENT_TRACK_ERR_DEG:.1f} deg.")
        else:
            notes.append(f"Pitch tracking miss fraction: {frac_bad*100:.1f}%.")

    # 4) Authority latch behavior
    if safe_col(flare_df, "flare_auth_limited"):
        auth_frac = (flare_df["flare_auth_limited"] > 0).mean()
        notes.append(f"Authority-limited active for {auth_frac*100:.1f}% of flare rows.")

    # 5) New elevator channels (if present)
    has_new = all(c in flare_df.columns for c in ["flare_pitch_in_cmd", "elev_defl_max_deg", "elev_defl_n"])
    if has_new:
        valid = flare_df[flare_df["elev_defl_n"] > 0]
        if not valid.empty:
            cmd_hi = valid[np.abs(valid["flare_pitch_in_cmd"]) >= 0.8]
            if not cmd_hi.empty:
                notes.append(f"High pitch-input samples (|cmd|>=0.8): {len(cmd_hi)} rows.")
            notes.append(
                "Elevator deflection stats in flare: "
                f"avg max|defl|={valid['elev_defl_max_deg'].mean():.2f} deg, "
                f"peak max|defl|={valid['elev_defl_max_deg'].max():.2f} deg."
            )
        else:
            notes.append("No elevator deflection samples found (elev_defl_n=0).")

print("=== Auto Findings ===")
if issues:
    for i, item in enumerate(issues, start=1):
        print(f"{i}. {item}")
else:
    print("No major flare anomalies flagged by current thresholds.")

print("\n=== Notes ===")
for item in notes:
    print(f"- {item}")


In [ ]:
# Optional: quick column explorer for troubleshooting schema changes
sorted(df.columns)
